# 3장 05. Kubernetes 핵심 개념 확인

이 notebook은 Kubernetes 명령어를 많이 실행하는 실습이 아닙니다. Container/Compose로 API를 확인한 뒤, 왜 Kubernetes가 필요한지와 어떤 상태를 확인해야 하는지만 짧게 정리합니다.


## 기본 개념: Linux host 위의 cluster control plane

Kubernetes는 Linux를 대체하는 새 운영체제가 아닙니다. 여러 Linux host 위에서 container를 어디에 실행할지, 원하는 상태와 실제 상태가 맞는지, 장애가 나면 어떻게 다시 맞출지를 관리하는 control plane입니다.

이번 강의에서 필요한 수준은 다음 네 가지입니다.

| 개념 | 쉬운 뜻 | 실습에서 확인할 것 |
| --- | --- | --- |
| desired state | Git/manifest에 선언한 원하는 상태 | Argo CD Application과 Kustomize path |
| live state | cluster 안에 실제 반영된 상태 | Argo CD sync, KServe Ready |
| controller | desired/live 차이를 계속 맞추는 loop | Application, InferenceService condition |
| scheduler | Pod를 어느 node에서 실행할지 정함 | Pod Pending/Running, event |
| etcd | cluster 상태 저장소 | 직접 조작하지 않고 object 상태로만 확인 |


## 1. Container와 Kubernetes 관심사 비교

아래 표는 이번 배포 실습에서 Container/Compose와 Kubernetes가 각각 어디까지 담당하는지 정리합니다.


In [ ]:
import pandas as pd

pd.DataFrame([
    {"stage": "Container", "question": "이미지가 실행 가능한가", "evidence": "Dockerfile, docker compose, /health"},
    {"stage": "Compose", "question": "API와 MLflow를 한 VM에서 함께 띄울 수 있는가", "evidence": "compose.yaml service, port, volume"},
    {"stage": "Kubernetes", "question": "원하는 배포 상태가 cluster에 반영되는가", "evidence": "Deployment, Service, InferenceService status"},
    {"stage": "Argo CD", "question": "Git의 desired state와 live state가 같은가", "evidence": "app diff, sync, health"},
])


출력에서 볼 핵심은 Kubernetes가 모델 품질을 자동으로 보장하지 않는다는 점입니다. Kubernetes는 실행과 상태 수렴을 담당하고, 모델 품질과 응답 의미는 MLflow 기록, FastAPI/KServe 응답, Grafana 로그와 metric으로 따로 확인합니다.


## 2. 이번 강의에서 Kubernetes 확인 범위

이번 과정에서는 Kubernetes를 깊게 운영하지 않습니다. 수강생이 남길 수 있어야 하는 말은 다음 정도입니다.


In [ ]:
checks = [
    "manifest에 MLflow Deployment와 Service가 있다",
    "manifest에 KServe InferenceService가 있다",
    "Argo CD Application이 Git path를 바라본다",
    "sync 전이면 live 배포 확인은 unverified로 남긴다",
    "Ready 확인 뒤에만 endpoint 응답과 Grafana 관측을 live evidence로 쓴다",
]
for index, item in enumerate(checks, start=1):
    print(f"{index}. {item}")


다음 notebook에서는 이 개념을 실제 manifest 파일에서 확인합니다. 먼저 MLflow가 Kubernetes에 배포될 준비가 되어 있는지 보고, 그 다음 KServe가 후보 모델 endpoint를 선언하는지 확인합니다.
